### Reseta arquivos de horarios

In [4]:
import json
import re
import os
import gdown
import fitz
import tempfile
import time
from io import BytesIO
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchAny, PayloadSchemaType

load_dotenv()

QDRANT_ENDPOINT = os.getenv("QDRANT_ENDPOINT")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
CEREBRAS_API_KEY = os.getenv("CEREBRAS_API_KEY")
COLLECTION_NAME = "ifrs-canoas"

client = QdrantClient(url=QDRANT_ENDPOINT, api_key=QDRANT_API_KEY)
cerebras_client = OpenAI(api_key=CEREBRAS_API_KEY, base_url="https://api.cerebras.ai/v1")

# URLs de download dos PDFs de horário
horario_urls_download = [
    "https://drive.google.com/uc?export=download&id=1giy2a2wZ6QMgynKqv997OjBDvp1f1WTl",
    "https://drive.google.com/uc?export=download&id=1iDkwv-NYtqDUq-fkrWeysMCqDWrcxnRV",
    "https://drive.google.com/uc?export=download&id=1kB8S27U-9gRUsC-fljykNMPxdk0nJvJm",
    "https://drive.google.com/uc?export=download&id=1u16bttceP7Hf1ampA-YeO7p3KF5hYa2h",
    "https://drive.google.com/uc?export=download&id=1RWYIwyjVJPhPotjFhz6hjymMQ2c71KUw",
    "https://drive.google.com/uc?export=download&id=16nyJTxh0MMXk8ThtqV57N-O7xeeSKqsG",
    "https://drive.google.com/uc?export=download&id=1v58tc9YycQ24oPeutaCtBW2Hah8GgNpp",
    "https://drive.google.com/uc?export=download&id=1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo",
    "https://drive.google.com/uc?export=download&id=1WppKds4GM3eH_UqquweiMlCl8o-Ty1fU",
    "https://drive.google.com/uc?export=download&id=1HuwnLNEK5lMjHZSr9sbmc62lzvziU8nh",
    "https://drive.google.com/uc?export=download&id=1zyh3AlzamjM7XppOCOPXD3WcIKHdTc9m",
]

def to_drive_view_url(url):
    if "drive.google.com/uc" in url:
        match = re.search(r"id=([a-zA-Z0-9_-]+)", url)
        if match:
            return f"https://drive.google.com/file/d/{match.group(1)}/view"
    return url

horario_urls_view = [to_drive_view_url(u) for u in horario_urls_download]

# remove do pdfs_parsed.json
with open("../data/raw/pdfs_parsed.json", "r", encoding="utf-8") as f:
    pdfs_parsed = json.load(f)

antes = len(pdfs_parsed)
pdfs_parsed = [p for p in pdfs_parsed if p["source_url"] not in set(horario_urls_download)]
print(f"pdfs_parsed: {antes} -> {len(pdfs_parsed)}")

with open("../data/raw/pdfs_parsed.json", "w", encoding="utf-8") as f:
    json.dump(pdfs_parsed, f, ensure_ascii=False, indent=2)

# remove do Qdrant
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="source_url",
    field_schema=PayloadSchemaType.KEYWORD
)

client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=Filter(
        must=[FieldCondition(key="source_url", match=MatchAny(any=horario_urls_view))]
    )
)

# confirma remoção
results = client.scroll(
    collection_name=COLLECTION_NAME,
    scroll_filter=Filter(
        must=[FieldCondition(key="source_url", match=MatchAny(any=horario_urls_view))]
    ),
    with_payload=True,
    limit=10
)
print(f"Pontos restantes no Qdrant: {len(results[0])}")
print("Limpeza concluída.")

pdfs_parsed: 595 -> 584
Pontos restantes no Qdrant: 0
Limpeza concluída.


### Retorna chunks a partir de link

In [3]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchAny, PayloadSchemaType

load_dotenv()

QDRANT_ENDPOINT = os.getenv("QDRANT_ENDPOINT")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "ifrs-canoas"

client = QdrantClient(url=QDRANT_ENDPOINT, api_key=QDRANT_API_KEY)

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="source_url",
    field_schema=PayloadSchemaType.KEYWORD
)

results = client.scroll(
    collection_name=COLLECTION_NAME,
    scroll_filter=Filter(
        must=[FieldCondition(
            key="source_url",
            match=MatchAny(any=["https://ifrs.edu.br/canoas/ensino/professores/"])
        )]
    ),
    with_payload=True,
    limit=10
)

print(f"Chunks encontrados: {len(results[0])}")
for p in results[0]:
    print(p.payload["text"][:])
    print("---")

Chunks encontrados: 2
Professores
Professores efetivos
⇔
A participação do docente no Núcleo de Integração do Ensino, Pesquisa e Extensão (NIEPE), conforme sua área de atuação, está disponível para consulta em:
https://ifrs.edu.br/canoas/institucional/niepe/
.
Professores efetivos
Adriana Braun
– Área: Física –
Currículo Lattes
Adriano Armando do Amarante
– Área: Filosofia –
Currículo Lattes
Adriel Mota Ziesemer Júnior
– Área: Informática Geral –
Currículo Lattes
Alexandre Tadachi Morey
– Área: Ciências Biológicas –
Currículo Lattes
Aline Noimann
– Área: Letras/Espanhol –
Currículo Lattes
Aline Santos Oliveira
– Área: Pedagogia –
Currículo Lattes
Ângelo Mozart Medeiros de Oliveira
– Área: Física –
Currículo Lattes
Augusto Alexandre Durgante de Mattos
– Área: Eletrônica –
Currículo Lattes
Bruno Brogni Uggioni –
Área: Matemática –
Currículo Lattes
Caio Graco Prates Alegretti
– Área: Matemática / Engenharia –
Currículo Lattes
Carina Loureiro Andrade
– Área: Matemática –
Currículo Lattes
C